# NRPy 2: A 10-Minute Overview

## Author: Zach Etienne

#### This notebook gives a compact tour of NRPy 2: symbolic tensor construction with Python and SymPy, optimized C code generation, and finite-difference-aware kernel output. The running example constructs the 18 independent spatial Christoffel symbols from a symmetric three-metric.

# Table of Contents

This notebook is organized as follows:

1. [Part 1](#Part-1:-Why-NRPy?): Why NRPy?
1. [Part 2](#Part-2:-Constructing-3-Christoffels-$\Gamma^i_{jk}$-as-symbolic-expressions-in-terms-of-3-metric-$\gamma_{ij}$-and-derivatives): Constructing 3-Christoffels $\Gamma^i_{jk}$ as symbolic expressions in terms of 3-metric $\gamma_{ij}$ and derivatives
1. [Part 3](#Part-3:--c_codegen()-example:-Output-$\Gamma^i_{jk}$-expressions-as-optimized-C-code,-assuming-derivatives-already-specified): `c_codegen()` example: Output $\Gamma^i_{jk}$ expressions as optimized C code, assuming derivatives already specified
1. [Part 4](#Part-4:--c_codegen()-example:-Specify-numerical-derivatives-within-$\Gamma^i_{jk}$-expressions-as-finite-differences): `c_codegen()` example: Specify numerical derivatives within $\Gamma^i_{jk}$ expressions as finite differences
1. [Part 5](#Part-5:-What-next?-Navigating-the-NRPy-2-torial): What next? Navigating the NRPy 2-torial

# Part 1: Why NRPy?
### \[Back to [top](#Table-of-Contents)\]

Numerical relativity often uses standard numerical methods, but the equations are large, tensorial, and difficult to implement by hand without errors.

[Einstein notation](https://en.wikipedia.org/wiki/Einstein_notation) makes this structure manageable on paper by suppressing explicit tensor sums.

NRPy 2 treats tensor equations written in Einstein notation as executable symbolic specifications. In Python,

* rank-0 tensors (scalars) are *variables*
* rank-1 tensors are *lists*
* rank-2 tensors are *lists of lists*
* ... and so forth.

Implied tensor summations become explicit Python loops.

NRPy combines this representation with [SymPy](https://www.sympy.org/) and core code-generation modules that convert symbolic expressions into optimized C kernels. Its finite-difference support can also insert stencil reads directly into generated code.

The NRPy 2-torial proceeds from setup through core APIs, finite differences, wave-equation examples, curvilinear coordinates, infrastructure targets, and numerical-relativity workflows. This overview focuses on the symbolic-to-C pipeline used throughout those notebooks.

# Part 2: Constructing 3-Christoffels $\Gamma^i_{jk}$ as symbolic expressions in terms of 3-metric $\gamma_{ij}$ and derivatives 
### \[Back to [top](#Table-of-Contents)\]

**Problem statement**: Given a three-metric $\gamma_{ij}$, construct the 18 independent Christoffel symbols $\Gamma^i_{jk}$. For now, assume that $\gamma_{ij}$ and its first derivatives are already available as symbols.

NRPy represents tensors and indexed expressions as nested Python lists, so for example

* $\gamma_{ij}=$ `gammaDD[i][j]`
* $\gamma_{ij,k}=$ `gammaDD_dD[i][j][k]`

Christoffel symbols are defined by

\begin{align}
\Gamma_{ij}^k &= \frac{1}{2} \gamma^{kl}\left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right)\\
\end{align}

First define $\gamma_{ij}$ and its inverse with NRPy's indexed-expression helpers.

In [ ]:
import sympy as sp
import nrpy.indexedexp as ixp

# gammaDD is a rank-2 indexed expression symmetric in indices 0 and 1.
gammaDD = ixp.declarerank2("gammaDD", symmetry="sym01", dimension=3)

# gammaUU is the inverse of gammaDD; gammaDET is its determinant.
gammaUU, gammaDET = ixp.symm_matrix_inverter3x3(gammaDD)

The generated objects encode the requested symmetry. The inverse metric is also symmetric, and $\gamma_{ij}\gamma^{ji}=3$ for a three-dimensional metric.

**Exercise**: Replace the final trace check below with explicit loops over all dimensions, then simplify the result.

In [ ]:
print("Check that gammaDD[0][1] = gammaDD[1][0]:")
print("gammaDD[0][1] = ", gammaDD[0][1])
print("gammaDD[1][0] = ", gammaDD[1][0])
assert gammaDD[0][1] == gammaDD[1][0]

print("\nOutput gammaUU[1][0] = ", gammaUU[1][0])
print("\nCheck that gammaUU[1][0] - gammaUU[0][1] = 0:")
difference = sp.simplify(gammaUU[1][0] - gammaUU[0][1])
print("gammaUU[1][0] - gammaUU[0][1] = ", difference)
assert difference == 0

trace = sp.simplify(sum(gammaDD[i][j] * gammaUU[j][i] for i in range(3) for j in range(3)))
print("\nTrace gammaDD[i][j] * gammaUU[j][i] = ", trace)
assert trace == 3

Define Christoffel symbols in terms of the inverse metric and metric first derivatives:
$$
\Gamma_{ij}^k = \frac{1}{2} \gamma^{kl}\left(\gamma_{jl,i} + \gamma_{il,j} - \gamma_{ij,l}\right)
$$

In [ ]:
# First define symbolic expressions for metric derivatives.
gammaDD_dD = ixp.declarerank3("gammaDD_dD", symmetry="sym01", dimension=3)

# Initialize GammaUDD (spatial Christoffel symbols) to zero.
GammaUDD = ixp.zerorank3(dimension=3)
for i in range(3):
    for j in range(3):
        for k in range(3):
            for l in range(3):
                GammaUDD[k][i][j] += sp.Rational(1, 2) * gammaUU[k][l] * (
                    gammaDD_dD[j][l][i]
                    + gammaDD_dD[i][l][j]
                    - gammaDD_dD[i][j][l]
                )

Now confirm the lower-index symmetry $\Gamma^k_{ij}=\Gamma^k_{ji}$.

In [ ]:
assert all(
    sp.simplify(GammaUDD[k][i][j] - GammaUDD[k][j][i]) == 0
    for i in range(3)
    for j in range(3)
    for k in range(3)
)
print("All GammaUDD[k][i][j] - GammaUDD[k][j][i] symmetry checks vanish.")

# Part 3:  `c_codegen()` example: Output $\Gamma^i_{jk}$ expressions as optimized C code, assuming derivatives already specified
### \[Back to [top](#Table-of-Contents)\]

At the core of NRPy is the ability to convert SymPy expressions to optimized C code.

**Problem statement**: Output all 18 unique Christoffel symbols with three optimization modes supported by NRPy's `c_codegen()`.

First we store all 18 unique Christoffel symbols, as well as their desired variable names in C, to Python lists.

In [ ]:
symbols_list = []
varname_list = []
for i in range(3):
    for j in range(3):
        for k in range(j, 3):
            symbols_list.append(GammaUDD[i][j][k])
            varname_list.append(f"ChristoffelUDD{i}{j}{k}")

assert len(symbols_list) == 18
assert len(varname_list) == 18
print(f"Prepared {len(symbols_list)} independent Christoffel expressions.")

Next pass these lists to NRPy's C code-generation function `c_codegen()`.

**Optimization Level 0**: compute each Christoffel symbol independently. This exposes repeated subexpressions.

In [ ]:
import nrpy.c_codegen as ccg


def print_code_excerpt(label, code, head_lines=0, markers=()):
    """Print selected lines from generated C code."""
    lines = code.splitlines()
    selected = list(range(min(head_lines, len(lines))))
    for marker in markers:
        for index, line in enumerate(lines):
            if marker in line:
                if index not in selected:
                    selected.append(index)
                break

    print(f"{label} ({len(selected)} of {len(lines)} selected lines shown):")
    previous = None
    for index in sorted(selected):
        if previous is not None and index > previous + 1:
            print("...")
        print(lines[index])
        previous = index


code_no_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=False,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
print_code_excerpt("No CSE", code_no_cse, markers=("ChristoffelUDD000",))

The unoptimized code recomputes many factors. Common subexpression elimination introduces temporaries for repeated symbolic subexpressions.

**Optimization Level 1**: use [common subexpression elimination (CSE)](https://en.wikipedia.org/wiki/Common_subexpression_elimination).

In [ ]:
code_cse = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
assert "ChristoffelUDD" in code_cse
print_code_excerpt("CSE", code_cse, head_lines=6, markers=("ChristoffelUDD000",))

**Optimization Level 2**: use CSE and [single instruction, multiple data (SIMD)](https://en.wikipedia.org/wiki/Single_instruction,_multiple_data) helper macros. This form is intended for kernels evaluated over numerical grid data.

In [ ]:
code_simd = ccg.c_codegen(
    symbols_list,
    varname_list,
    enable_cse=True,
    enable_simd=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)
print_code_excerpt(
    "CSE plus SIMD", code_simd, head_lines=14, markers=("ChristoffelUDD000",)
)

# Part 4:  `c_codegen()` example: Specify numerical derivatives within $\Gamma^i_{jk}$ expressions as finite differences
### \[Back to [top](#Table-of-Contents)\]

The previous C code assumed that $\gamma_{ij,k}$ values had already been computed. NRPy's finite-difference-aware code generation can instead insert stencil expressions for derivative symbols.

Here we use a uniform-grid finite-difference example. The input metric components are registered as gridfunctions, and the independent Christoffel components are registered as output gridfunctions.

The parameter `fd_order=6` selects sixth-order finite-difference stencils. `Infrastructure="BHaH"` selects the memory-access convention used in this generated kernel.

The example keeps CSE enabled and omits SIMD so the stencil structure remains compact enough to inspect.

In [ ]:
import nrpy.grid as gri
import nrpy.params as par

gri.glb_gridfcs_dict.clear()
par.set_parval_from_str("Infrastructure", "BHaH")
par.set_parval_from_str("fd_order", 6)

gri.register_gridfunctions_for_single_rank2(
    "gammaDD", group="EVOL", dimension=3, symmetry="sym01"
)

christoffel_gf_names = list(varname_list)
gri.register_gridfunctions(
    christoffel_gf_names,
    group="AUX",
    dimension=3,
    rank=3,
    is_basename=False,
)

output_varnames = [
    gri.BHaHGridFunction.access_gf(gf_name=name, gf_array_name="aux_gfs")
    for name in christoffel_gf_names
]
fd_code = ccg.c_codegen(
    symbols_list,
    output_varnames,
    enable_cse=True,
    enable_fd_codegen=True,
    verbose=False,
    enable_clang_format=False,
    include_braces=False,
)

for marker in [
    "NRPy-Generated GF Access/FD Code",
    "IDX4",
    "aux_gfs[IDX4(CHRISTOFFELUDD000GF",
    "gammaDD_dD000",
]:
    assert marker in fd_code, marker

print_code_excerpt(
    "Finite-difference C code",
    fd_code,
    head_lines=12,
    markers=("gammaDD_dD000", "aux_gfs[IDX4(CHRISTOFFELUDD000GF"),
)

# Part 5: What next? Navigating the NRPy 2-torial
### \[Back to [top](#Table-of-Contents)\]

Continue with the notebooks most relevant to your next task:

- [Repository index](../index.ipynb)
- [Package map](../0-getting_started/package_map.ipynb)
- [Wave equation with NumPy](wave_equation_with_numpy.ipynb)
- [C code generation](c_codegen.ipynb)
- [Parameters](params.ipynb)
- [Gridfunctions](grid.ipynb)
- [Indexed expressions](indexedexp.ipynb)
- [Finite differences](finite_difference.ipynb)
- [Wave equation with C code generation](../2-basic_physics_applications/wave_equation_and_c_codegen.ipynb)